# Model animations

Animated and interactive views of the four Kitaev-chain models, built on a
completed `four_model_comparison.py` run. Nothing here trains anything.

The numerics live in `kitaev.visualisation.animation`; this notebook wires
them onto a run's checkpoints. Particle and hole sectors are kept apart
throughout, since their balance is the physical signature of a Majorana
end mode.

**What the raw views mean.** Outside the topological phase the lowest
state is non-degenerate and the raw per-sector density `|psi^p_n|^2` /
`|psi^h_n|^2` is a genuine per-site test. Inside it (`|mu| < 2t`) the
`+-lambda_1` pair is degenerate and the folded-spectrum objective is flat
over it, so *which* representative of the pair a run lands on -- and hence
its raw particle/hole split -- is a gauge choice, not physics. A model can
have the energy, gap, subspace fidelity and the gauge-invariant pair
density `rho_n/2` all correct and still show a raw split that jumps
between seeds and frames. Sections 1-3 make that gauge freedom visible;
section 4 pins down what survives it.

Sections:

1. Particle / hole density and residual, swept over the chemical potential.
2. Interactive chemical-potential explorer.
3. Cross-seed fan, one model at a time, saved per model.
4. Cross-seed density without the animation: the publication figures.
5. Spectrum movie, the lowest distinct levels drawing themselves out.
6. Save the set as gifs next to the run.

## Setup

The comparison script is a sibling module under `experiments/`, so put that
directory on the path before importing it.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the experiments package is visible."""
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "four_model_comparison.py").exists():
            return candidate
    raise RuntimeError(f"could not locate the repo root starting from {start}")


REPO = _find_repo_root(Path.cwd())
if str(REPO / "experiments") not in sys.path:
    sys.path.insert(0, str(REPO / "experiments"))

In [ ]:
import numpy as np
import torch
from four_model_comparison import DELTA, MODEL_ORDER, RECIPES, N, T
from IPython.display import HTML
from ipywidgets import Dropdown, FloatSlider, interact
from matplotlib import pyplot as plt

from kitaev.analytical import KitaevChainHamiltonian
from kitaev.visualisation import (
    MovieModel,
    animate_seed_fan,
    animate_spectrum,
    animate_wavefunction_residual,
    build_wavefunction_movie,
    save_animation,
    use_house_style,
)
from kitaev.visualisation.style import CORAL, INK
from kitaev.xai.loading import load_seed_checkpoints, psi_only

# to_jshtml() embeds every frame as a PNG; lift the default 20 MB cap.
plt.rcParams["animation.embed_limit"] = 80.0

In [ ]:
# --- configuration ----------------------------------------------------
_sessions = sorted((REPO / "results" / "logs").glob("*four-model-comparison"))
RUN_DIR = _sessions[-1] if _sessions else None
DEVICE = "cpu"
MODELS = list(MODEL_ORDER)
MU_FRAMES = np.linspace(-4.0, 4.0, 61)  # ~4 s at 15 fps
ANIM_DIR = None if RUN_DIR is None else RUN_DIR / "xai" / "animations"

assert RUN_DIR is not None, "no four-model-comparison session under results/logs"
print("run dir :", RUN_DIR)

hamiltonian = KitaevChainHamiltonian(n_sites=N, hopping=T, pairing=DELTA)


def _movie_model(name, model):
    recipe = RECIPES[name]
    residual_model = (
        model if recipe.basis == "chiral" else psi_only(model, device=DEVICE)
    )
    return MovieModel(
        recipe.plot_label, recipe.build_adapt(model), residual_model, recipe.basis
    )


# one checkpoint list per model, kept for reuse across the sections
seed_checkpoints = {
    name: load_seed_checkpoints(
        RECIPES[name].build_model, RUN_DIR / "checkpoints" / name, device=DEVICE
    )
    for name in MODELS
}
movie_models = [_movie_model(name, seed_checkpoints[name][0]) for name in MODELS]
for name in MODELS:
    print(f"  {RECIPES[name].plot_label:<18} {len(seed_checkpoints[name])} seeds")

## 1. Particle / hole density and residual

Left two panels: the particle-sector density `|psi^p_n(mu)|^2` and the
hole-sector density `|psi^h_n(mu)|^2` for every model, with the exact
profile filled behind. A good Majorana end mode has the two panels
agreeing site by site *in the trivial phase*; inside `|mu| < 2t` the raw
split shown here is gauge-dependent (see the intro), so read it as a view
of the gauge freedom rather than a pass/fail against the exact curve.
Right panel: each model's pointwise physics residual with a playhead at
the current `mu`.

In [ ]:
movie = build_wavefunction_movie(movie_models, hamiltonian, MU_FRAMES, device=DEVICE)
wave_anim = animate_wavefunction_residual(movie, hopping=T, fps=15, dpi=64)
HTML(wave_anim.to_jshtml())

## 2. Interactive explorer

Drag `mu` and switch models. Useful for parking in the spike region near
`mu / t = +-0.4` and studying one model's particle / hole balance by hand.

In [ ]:
_sites = np.arange(N)
_by_label = {m.label: m for m in movie_models}


def _explore(mu=0.4, model_label=movie_models[0].label):
    use_house_style()
    entry = _by_label[model_label]
    _, eigenvectors = np.linalg.eigh(hamiltonian.build(float(mu)))
    psi_exact = eigenvectors[:, N]
    with torch.no_grad():
        mu_t = torch.tensor([[float(mu)]], dtype=torch.float32)
        psi = entry.adapter(mu_t)[1].numpy().ravel()
    psi = psi / np.linalg.norm(psi)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.0), sharey=True)
    for ax, sector, exact, pred in (
        (axes[0], r"particle $|\psi^p_n|^2$", psi_exact[:N] ** 2, psi[:N] ** 2),
        (axes[1], r"hole $|\psi^h_n|^2$", psi_exact[N:] ** 2, psi[N:] ** 2),
    ):
        ax.fill_between(_sites, exact, color=INK, alpha=0.14, lw=0, label="exact")
        ax.plot(_sites, pred, color=CORAL, lw=1.8, ls=(0, (3, 2)), label=model_label)
        ax.set_xlabel("site $n$")
        ax.set_title(sector)
    phase = "topological" if abs(mu) < 2 * T else "trivial"
    axes[0].set_ylabel(r"$|\psi_n|^2$")
    axes[0].legend(loc="upper center")
    fig.suptitle(rf"$\mu = {mu:+.2f}\,t$  ({phase})")
    plt.show()


interact(
    _explore,
    mu=FloatSlider(min=-4.0, max=4.0, step=0.05, value=0.4, description="mu / t"),
    model_label=Dropdown(
        options=[m.label for m in movie_models], value=movie_models[0].label
    ),
)

## 3. Cross-seed fan

Every seed's particle and hole densities for a model, swept over the full
`mu` range, each seed its own colour. Where the model is under-determined
the seed curves fan apart and jitter frame to frame; where it is pinned
they collapse onto one line.

Inside `|mu| < 2t` the fan of the three Nambu-basis models (FSM baseline,
semi-supervised, structural Nambu) is the degenerate `+-lambda_1` pair
being resolved differently by each seed -- a gauge degree of freedom the
folded-spectrum loss never fixes, *not* a failure to solve the physics.
Every gauge-invariant quantity (energy, gap, subspace fidelity, the pair
density `rho_n/2`) is still recovered; section 4 shows that companion
view. Only the chiral model pins a representative structurally -- its
`N x N` residual has no flat direction -- so only its fan stays tight.

One animation per model, each also written to
`<run>/xai/animations/seed_fan_<model>.gif`.

In [ ]:
_ref = build_wavefunction_movie(movie_models[:1], hamiltonian, MU_FRAMES, device=DEVICE)

fan_anims = {}
for name in MODELS:
    recipe = RECIPES[name]
    models = seed_checkpoints[name]
    if not models:
        print(f"skip {name}: no checkpoints")
        continue
    per_seed = [
        build_wavefunction_movie(
            [_movie_model(name, m)], hamiltonian, MU_FRAMES, device=DEVICE
        )
        for m in models
    ]
    fan_anims[name] = animate_seed_fan(
        [mv.particle[recipe.plot_label] for mv in per_seed],
        [mv.hole[recipe.plot_label] for mv in per_seed],
        MU_FRAMES,
        model_label=recipe.plot_label,
        transition=2 * T,
        particle_exact=_ref.particle_exact,
        hole_exact=_ref.hole_exact,
        fps=15,
        dpi=80,
        save_path=ANIM_DIR / f"seed_fan_{name}.gif",
    )
    print("wrote", ANIM_DIR / f"seed_fan_{name}.gif")

Preview each model's fan inline. Only the chiral model stays tight across
the whole sweep; the three Nambu-basis models -- FSM baseline,
semi-supervised and structural Nambu -- fan out inside `|mu| < 2t`. That
fanning is expected: it is the unresolved gauge angle within the
degenerate pair, not a wrong answer, and section 4 quantifies how little
of it survives in the gauge-invariant density.

In [ ]:
HTML(fan_anims["structural_nambu"].to_jshtml())

In [ ]:
HTML(fan_anims["chiral"].to_jshtml())

In [ ]:
HTML(fan_anims["nambu_baseline"].to_jshtml())

In [ ]:
HTML(fan_anims["semi_supervised"].to_jshtml())

## 4. Cross-seed density, without the animation

The same finding as the fan above, as static publication figures.
`sweep_seed_densities` stacks every seed of one model over a dense `mu`
grid in two parallel views:

- the **raw** per-sector density `|psi^p_n|^2` / `|psi^h_n|^2` -- a gauge
  choice inside `|mu| < 2t`, so the seeds scatter there;
- the **gauge-invariant** pair density `rho_n/2`, the projector diagonal
  of `span{psi, Xi psi}` -- basis-free, so it collapses onto the exact
  curve for every model that found the right near-zero subspace, the
  Nambu-basis models included.

Three figures: per-model dispersion maps (`site` x `mu` inter-seed std),
per-model fixed-`mu` seed slices, and a shared left-end-weight envelope.
`four_model_comparison.py --figures-only` writes the same set into
`<run>/figures/comparison/`.

In [ ]:
from kitaev.visualisation import (
    plot_seed_density_dispersion_maps,
    plot_seed_density_slices,
    plot_seed_edge_weight_envelope,
    sweep_seed_densities,
)

_dense_mu = np.linspace(-4.0, 4.0, 160)
seed_density = {
    name: sweep_seed_densities(
        [RECIPES[name].build_adapt(m) for m in seed_checkpoints[name]],
        hamiltonian,
        _dense_mu,
        model_label=RECIPES[name].plot_label,
        device=DEVICE,
    )
    for name in MODELS
    if len(seed_checkpoints[name]) >= 2
}

for _fan in seed_density.values():
    plot_seed_density_dispersion_maps(_fan, hopping=T)
    plot_seed_density_slices(_fan, hopping=T)
plot_seed_edge_weight_envelope(list(seed_density.values()), hopping=T)
plt.show()

## 5. Spectrum movie

The lowest distinct exact levels `sigma_k(mu)` drawn out as `mu` sweeps.
`sigma_1` collapses towards zero for `|mu| < 2t` and the bulk gap closes at
`|mu| = 2t`.

In [ ]:
spectrum_anim = animate_spectrum(hamiltonian, MU_FRAMES, n_levels=4, fps=15, dpi=90)
HTML(spectrum_anim.to_jshtml())

## 6. Save the set

The per-model fans are already on disk from section 3; this writes the
wavefunction and spectrum animations alongside them.

In [ ]:
for name, anim in (
    ("wavefunction_residual", wave_anim),
    ("spectrum", spectrum_anim),
):
    print(save_animation(anim, ANIM_DIR / f"{name}.gif", fps=15))
for name in fan_anims:
    print(ANIM_DIR / f"seed_fan_{name}.gif")